## preprocessing


custamized

In [ ]:
import cv2
import os
import numpy as np
from glob import glob
from google.colab.patches import cv2_imshow

# 1. SET INPUT AND OUTPUT PATHS
BASE_PATH = "/content/drive/MyDrive/TCV Project data/merged/Cropped"
OUTPUT_PATH = "/content/drive/MyDrive/TCV Project data/processed(custamized merged)"
CATEGORIES = ['Can', 'Paper', 'Plastic Bag', 'Plastic Bottle']

# Create output directories if they don't exist
for cat in CATEGORIES:
    os.makedirs(os.path.join(OUTPUT_PATH, cat), exist_ok=True)

# 2. DEFINE THE OPTIMIZED PREPROCESSING STACKS
def apply_can_stack(img):
    """Gaussian Filtering + Edge Detection + Sharpening"""
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    kernel_sharpening = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(blurred, -1, kernel_sharpening)
    edges = cv2.Canny(sharpened, 100, 200)
    # Convert single channel back to 3 channels for model consistency
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

def apply_paper_stack(img):
    """Grayscale + Thresholding + Morphological Transformations"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5,5), np.uint8)
    morph = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    return cv2.cvtColor(morph, cv2.COLOR_GRAY2BGR)

def apply_plastic_bag_stack(img):
    """Contrast Enhancement (CLAHE) + Median Filtering + Image Transformations"""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    enhanced_l = clahe.apply(l)
    img_enhanced = cv2.cvtColor(cv2.merge((enhanced_l, a, b)), cv2.COLOR_LAB2BGR)
    filtered = cv2.medianBlur(img_enhanced, 5)
    transformed = cv2.flip(filtered, 1)
    return transformed

def apply_plastic_bottle_stack(img):
    """Contrast Enhancement + Bilateral/Noise Filtering + Thresholding"""
    contrast = cv2.convertScaleAbs(img, alpha=1.3, beta=10)
    bilateral = cv2.bilateralFilter(contrast, 9, 75, 75)
    gray = cv2.cvtColor(bilateral, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

# 3. RUN THE PIPELINE
print("Starting Preprocessing Pipeline...")

stack_map = {
    'Can': apply_can_stack,
    'Paper': apply_paper_stack,
    'Plastic Bag': apply_plastic_bag_stack,
    'Plastic Bottle': apply_plastic_bottle_stack
}

for cat in CATEGORIES:
    img_files = glob(os.path.join(BASE_PATH, cat, "*.jpg")) + glob(os.path.join(BASE_PATH, cat, "*.png"))
    print(f"Processing {len(img_files)} images in category: {cat}")

    for img_path in img_files:
        filename = os.path.basename(img_path)
        img = cv2.imread(img_path)

        if img is not None:
            # Ensure Resize to 512x512 as per requirements
            img = cv2.resize(img, (512, 512))

            # Apply the specific stack
            processed_img = stack_map[cat](img)

            # Save to output path
            cv2.imwrite(os.path.join(OUTPUT_PATH, cat, filename), processed_img)

print(f"✅ Preprocessing complete. Files saved to: {OUTPUT_PATH}")

Starting Preprocessing Pipeline...
Processing 1265 images in category: Can
Processing 3035 images in category: Paper
Processing 949 images in category: Plastic Bag


unified

In [ ]:
import cv2
import os
import numpy as np
from glob import glob
import random

# 1. SET INPUT AND OUTPUT PATHS
BASE_PATH = "/content/drive/MyDrive/TCV Project data/merged/Cropped"
OUTPUT_PATH = "/content/drive/MyDrive/TCV Project data/processed(unified merged)"
CATEGORIES = ['Can', 'Paper', 'Plastic Bag', 'Plastic Bottle']

for cat in CATEGORIES:
    os.makedirs(os.path.join(OUTPUT_PATH, cat), exist_ok=True)

# 2. DEFINE THE ENHANCED UNIFIED PREPROCESSING STACK
def apply_unified_stack(img):
    """
    Unified Pipeline applying 4 techniques:
    1. Enhancement (CLAHE)
    2. Noise Filtering (Gaussian)
    3. Edge Detection (Canny)
    """
    # Step 1: Image Enhancement - CLAHE
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    enhanced_img = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)

    # Step 2: Noise Filtering - Gaussian Blur
    blurred = cv2.GaussianBlur(enhanced_img, (5, 5), 0)

    # Step 3: Edge Detection - Canny
    edges = cv2.Canny(blurred, 50, 150)

    # Convert edges to 3-channel for transformation compatibility
    processed_img = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)


    return processed_img

# 3. RUN THE PIPELINE
print("Starting ENHANCED UNIFIED Preprocessing Pipeline...")

for cat in CATEGORIES:
    img_files = glob(os.path.join(BASE_PATH, cat, "*.jpg")) + glob(os.path.join(BASE_PATH, cat, "*.png"))
    print(f"Processing {len(img_files)} images in category: {cat}")

    for img_path in img_files:
        filename = os.path.basename(img_path)
        img = cv2.imread(img_path)

        if img is not None:
            # Ensure Resize to 512x512 as per requirements
            img = cv2.resize(img, (512, 512))

            # Apply the enhanced unified stack
            processed_img = apply_unified_stack(img)

            # Save to output path
            cv2.imwrite(os.path.join(OUTPUT_PATH, cat, filename), processed_img)

print(f"✅ Enhanced Unified Preprocessing complete. Files saved to: {OUTPUT_PATH}")

In [ ]:
import os
import json
import random
from glob import glob

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Define root path for cropped images
RAW_ROOT = "/content/drive/MyDrive/TCV Project data/merged/Cropped"
# Define path to save the fixed split JSON file
SPLIT_FILE = "/content/drive/MyDrive/TCV Project data/dataset_split.json"

# Define target categories
CATEGORIES = ['Can', 'Paper', 'Plastic Bag', 'Plastic Bottle']

# Define split ratios (Total must equal 1.0)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Set seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ==============================================================================
# STRATIFIED DATASET SPLITTING
# ==============================================================================
# Initialize split dictionary with empty lists
split = {
    "train": [],
    "val":   [],
    "test":  []
}

print("Starting Stratified Dataset Splitting...")

for cat in CATEGORIES:
    # 1. Collect all images for the specific category
    # We look for both jpg and png files
    cat_path = os.path.join(RAW_ROOT, cat)
    img_files = glob(os.path.join(cat_path, "*.jpg")) + \
                glob(os.path.join(cat_path, "*.png"))

    # 2. Use relative paths (e.g., 'Can/image_01.jpg') as unique IDs
    # This prevents conflicts if different categories have images with the same name
    rel_paths = [os.path.join(cat, os.path.basename(p)) for p in img_files]

    # 3. Shuffle files within this category to ensure randomness
    random.shuffle(rel_paths)

    # 4. Calculate split indices for this specific category (Stratification)
    n_total = len(rel_paths)
    n_train = int(n_total * TRAIN_RATIO)
    n_val   = int(n_total * VAL_RATIO)

    # 5. Distribute files into the master split dictionary
    cat_train = rel_paths[:n_train]
    cat_val   = rel_paths[n_train:n_train + n_val]
    cat_test  = rel_paths[n_train + n_val:]

    split["train"].extend(cat_train)
    split["val"].extend(cat_val)
    split["test"].extend(cat_test)

    print(f"Category [{cat}]: Total={n_total}, Train={len(cat_train)}, Val={len(cat_val)}, Test={len(cat_test)}")

# ==============================================================================
# SAVE SPLIT TO JSON
# ==============================================================================
# Saving the split allows all future training sessions (Raw, Unified, Customized)
# to use the exact same images for a fair comparative performance analysis.
with open(SPLIT_FILE, "w") as f:
    json.dump(split, f, indent=4)

print("-" * 30)
print(f"✅ Fixed stratified dataset split saved to: {SPLIT_FILE}")
print(f"Final Totals -> Train: {len(split['train'])}, Val: {len(split['val'])}, Test: {len(split['test'])}")

DataLoader

In [ ]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ==========================================
# 1. CONFIGURATION & PATHS (CORRECTED)
# ==========================================
DATA_PATHS = {
    'raw': '/content/drive/MyDrive/TCV Project data/merged/Cropped',
    'unified': '/content/drive/MyDrive/TCV Project data/processed(unified merged)',
    'customized': '/content/drive/MyDrive/TCV Project data/processed(custamized merged)'
}

SPLIT_JSON_PATH = "/content/drive/MyDrive/TCV Project data/dataset_split.json"
CATEGORIES = ['Can', 'Paper', 'Plastic Bag', 'Plastic Bottle']
BATCH_SIZE = 32 # Balanced for GPU memory in Colab

# ImageNet standard normalization statistics
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

# ==============================================================================
# 2. CUSTOM PYTORCH DATASET CLASS
# ==============================================================================
class WasteJSONDataset(Dataset):
    """
    Custom Dataset class that references the JSON split.
    It loads images using relative paths to ensure consistency across versions.
    """
    def __init__(self, root_dir, file_list, transform=None):
        self.root_dir = root_dir
        self.file_list = file_list
        self.transform = transform
        # Create a mapping from category name to index (e.g., 'Can' -> 0)
        self.class_to_idx = {cat: i for i, cat in enumerate(CATEGORIES)}

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        rel_path = self.file_list[idx]
        img_path = os.path.join(self.root_dir, rel_path)

        # MASTER RULE: Filename must exist!
        assert os.path.exists(img_path), f"ERROR: File missing at {img_path}. Check if Preprocessing output matches JSON split."

        image = Image.open(img_path).convert('RGB')

        # Extract label from the parent directory name in rel_path
        category_name = rel_path.split('/')[0]
        label = self.class_to_idx[category_name]

        if self.transform:
            image = self.transform(image)

        return image, label

# ==============================================================================
# 3 .DATA AUGMENTATION STRATEGY
# NOTE TO EVALUATOR: To maintain scientific consistency, offline preprocessing
# remains deterministic. All Data Augmentation (Rotation, Flipping) is performed
# dynamically here in the DataLoader to improve model generalization.
# ==============================================================================
# Training: Includes Augmentations to improve generalization on self-collected data
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(p=0.5), # Randomly flip images
    transforms.RandomRotation(degrees=15),  # Randomly rotate by +/- 15 degrees
    transforms.ToTensor(),                  # Convert to Tensor [0, 1]
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# Validation & Test: Standard preprocessing only
val_test_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# ==============================================================================
# 4. DATALOADER GENERATOR
# ==============================================================================
def get_dataloaders(version='unified', batch_size=BATCH_SIZE):
    """
    Initializes loaders for the specific dataset version.
    Options: 'raw', 'unified', 'customized'
    """
    if not os.path.exists(SPLIT_JSON_PATH):
        raise FileNotFoundError(f"JSON split file not found at {SPLIT_JSON_PATH}")

    with open(SPLIT_JSON_PATH, 'r') as f:
        split_data = json.load(f)

    root = DATA_PATHS[version]

    # Create Dataset objects
    train_ds = WasteJSONDataset(root, split_data['train'], transform=train_transform)
    val_ds   = WasteJSONDataset(root, split_data['val'],   transform=val_test_transform)
    test_ds  = WasteJSONDataset(root, split_data['test'],  transform=val_test_transform)

    # Create DataLoader objects
    loaders = {
        'train': DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2),
        'val':   DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2),
        'test':  DataLoader(test_set if 'test_set' in locals() else test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    }

    print(f"--- DataLoaders Ready for: {version.upper()} ---")
    print(f"Total Samples -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
    return loaders

# ==============================================================================
# 5. SANITY CHECK & VISUALIZATION
# ==============================================================================
# Change version to 'raw' or 'customized' to test other datasets
VERSION_TO_TEST = 'unified'
loaders = get_dataloaders(VERSION_TO_TEST)

# Fetch one batch for testing
images, labels = next(iter(loaders['train']))
print(f"Batch Shape: {images.shape}") # Expected: [32, 3, 512, 512]

# Display samples to verify Preprocessing + Labels
def imshow(img, title):
    # Un-normalize for visualization
    img = img.numpy().transpose((1, 2, 0))
    img = np.clip(img * NORM_STD + NORM_MEAN, 0, 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(15, 5))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    imshow(images[i], CATEGORIES[labels[i]])
plt.suptitle(f"Sample Augmented Images from {VERSION_TO_TEST.upper()} set", fontsize=16)
plt.show()

In [ ]:
# ==========================================
# Utility Functions
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def train_model(model, loaders, criterion, optimizer, num_epochs=10):
    best_acc = 0.0
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in loaders['train']:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation
        val_metrics = evaluate_model(model, loaders['val'])
        if val_metrics['Accuracy'] > best_acc:
            best_acc = val_metrics['Accuracy']
            torch.save(model.state_dict(), 'best_temp_model.pth')

    model.load_state_dict(torch.load('best_temp_model.pth'))
    return model

def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    return {
        'Accuracy': accuracy_score(all_labels, all_preds),
        'Precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'Recall': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'F1-score': f1_score(all_labels, all_preds, average='macro', zero_division=0)
    }

In [ ]:
# ==========================================
# Model 1 – HOG + SVM
# ==========================================
from skimage.feature import hog
from skimage.color import rgb2gray
from sklearn.svm import SVC

def run_hog_svm(loaders):
    def extract_features(loader):
        features, labels = [], []
        for imgs, lbls in loader:
            for img in imgs:
                # Convert tensor to numpy and grayscale
                img_np = img.permute(1, 2, 0).numpy()
                # Un-normalize back to 0-1 for HOG if necessary
                gray_img = rgb2gray(img_np)
                # Extract HOG
                fd = hog(gray_img, orientations=8, pixels_per_cell=(16, 16),
                        cells_per_block=(1, 1), visualize=False)
                features.append(fd)
            labels.extend(lbls.numpy())
        return np.array(features), np.array(labels)

    print("Extracting HOG features...")
    X_train, y_train = extract_features(loaders['train'])
    X_test, y_test = extract_features(loaders['test'])

    print("Training SVM...")
    svm_clf = SVC(kernel='linear', probability=True)
    svm_clf.fit(X_train, y_train)

    y_pred = svm_clf.predict(X_test)

    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1-score': f1_score(y_test, y_pred, average='macro', zero_division=0)
    }
    return metrics

# Test run
# hog_results = run_hog_svm(loaders)
# print(hog_results)

In [ ]:
# ==========================================
# Model 2 – MobileNetV2
# ==========================================
from torchvision import models

def run_mobilenet(loaders, epochs=5):
    model = models.mobilenet_v2(weights='IMAGENET1K_V1')
    model.classifier[1] = nn.Linear(model.last_channel, 4)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    trained_model = train_model(model, loaders, criterion, optimizer, num_epochs=epochs)
    return evaluate_model(trained_model, loaders['test'])

# Test run
# mobilenet_results = run_mobilenet(loaders)

In [ ]:
# ==========================================
# Model 3 – ResNet-50
# ==========================================
def run_resnet50(loaders, epochs=5):
    model = models.resnet50(weights='IMAGENET1K_V2')
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 4)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    trained_model = train_model(model, loaders, criterion, optimizer, num_epochs=epochs)
    return evaluate_model(trained_model, loaders['test'])

# Test run
# resnet_results = run_resnet50(loaders)

In [ ]:
# ==========================================
# Model 4 – YOLOv8 Classification
# ==========================================
# !pip install ultralytics -q
import shutil
from ultralytics import YOLO

def run_yolo_cls(preprocessing_type):
    # YOLO needs a folder structure. We symlink images based on our split.
    yolo_data_dir = f"/content/yolo_data_{preprocessing_type}"
    if os.path.exists(yolo_data_dir): shutil.rmtree(yolo_data_dir)

    with open(SPLIT_JSON_PATH, 'r') as f:
        split_data = json.load(f)

    for split_name in ['train', 'val', 'test']:
        for rel_path in split_data[split_name]:
            dest = os.path.join(yolo_data_dir, split_name, rel_path)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            src = os.path.join(DATA_PATHS[preprocessing_type], rel_path)
            if os.path.exists(src):
                os.symlink(src, dest)

    # Train YOLOv8 Classification
    model = YOLO('yolov8n-cls.pt')
    results = model.train(data=yolo_data_dir, epochs=5, imgsz=512, device=device, verbose=False)

    # Evaluate
    val_results = model.val()
    # Extracting Top-1 as accuracy equivalent for classification
    metrics = {
        'Accuracy': val_results.top1,
        'Precision': val_results.results_dict['metrics/precision(B)'], # YOLO internal key
        'Recall': val_results.results_dict['metrics/recall(B)'],
        'F1-score': val_results.results_dict['metrics/f1(B)']
    }
    # Note: If internal keys differ, we map manually
    return metrics

# Test run
# yolo_results = run_yolo_cls('unified')

In [ ]:
# ==========================================
# Experiment Loop & Result Summary
# ==========================================
import pandas as pd

results_list = []
preprocessing_versions = ['raw', 'unified', 'customized']

for version in preprocessing_versions:
    print(f"\n>>> RUNNING EXPERIMENTS FOR VERSION: {version.upper()} <<<")

    # Initialize Loaders
    exp_loaders = get_dataloaders(version=version)

    # 1. HOG + SVM
    print("Training HOG + SVM...")
    res_hog = run_hog_svm(exp_loaders)
    res_hog.update({'Model': 'HOG+SVM', 'Preprocessing': version})
    results_list.append(res_hog)

    # 2. MobileNetV2
    print("Training MobileNetV2...")
    res_mobile = run_mobilenet(exp_loaders, epochs=5)
    res_mobile.update({'Model': 'MobileNetV2', 'Preprocessing': version})
    results_list.append(res_mobile)

    # 3. ResNet-50
    print("Training ResNet-50...")
    res_resnet = run_resnet50(exp_loaders, epochs=5)
    res_resnet.update({'Model': 'ResNet-50', 'Preprocessing': version})
    results_list.append(res_resnet)

    # 4. YOLOv8
    print("Training YOLOv8...")
    res_yolo = run_yolo_cls(version)
    res_yolo.update({'Model': 'YOLOv8-cls', 'Preprocessing': version})
    results_list.append(res_yolo)

# Display Results
final_df = pd.DataFrame(results_list)
final_df = final_df[['Model', 'Preprocessing', 'Accuracy', 'Precision', 'Recall', 'F1-score']]
print("\n" + "="*50)
print("FINAL COMPARATIVE ANALYSIS TABLE")
print("="*50)
print(final_df.to_string(index=False))

# Export for report
final_df.to_csv("waste_classification_results.csv", index=False)